### <u><center><span style="color:darkorchid">ChatGPT</span> Resume Optimizer Final</u></center>

#### This will be the deployable version of the <span style="color:orange">original</span> <span style="color:darkorchid">ChatGPT </span><span style="color:orange">notebook</span>. The main differences here are:
> ##### 1.) Instead of creating paths to my personal machine, the program will create paths to the user's own machine; and
> ##### 2.) The back-end of the button <span style="color:deeppink">Company Name</span> will be created here as well. The point of this button is to include the hiring company's name in the title of your saved resume for facilitated resume tracking.

#### <u><span style="color:green">Imports</span></u>:
##### Because we're making the changes listed above, two additional Python librarys are imported in this section: <b>uuid</b> and <b>re</b>:
> ##### <b><u>uuid</u></b> - generates unique IDs so multiple users can upload files and download tailored resumes with unique filenames.
> ##### <b><u>re</u></b> - the regular expressions library lets us take the company name from the one the user entered and removes all unnecessary spaces for a cleaner filename.

In [1]:
import os # use personal openai key
import uuid # for unique file usernames
import re # for future 'company name' button
from dotenv import load_dotenv # loads environmental variables

# for working with OpenAI:
from openai import OpenAI

# HTML to Markdown for editing
from markdown import markdown
from IPython.display import display, Markdown # <-- make it look nice in the notebook

# for design
from weasyprint import HTML # see previous notebook for weasyprint notes

In [2]:
# load the environment variables
load_dotenv()

True

In [3]:
# make a destination 'gpt_resumes' directory for the work
os.makedirs("gpt_resumes", exist_ok=True)

***
#### Step 1: Open and Read the Markdown Resume
***

In [5]:
file_path = input("Please paste the full path to your resume file (eg: '/Users/your_name/Desktop/your_resume.md'): ").strip()

with open(file_path, "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

Please paste the full path to your resume file (eg: '/Users/your_name/Desktop/your_resume.md'):  /Users/nicholasjoseph/Desktop/ResumesEtc/nj_master_resume.md


#### Step 1a: Open and Read PDF Resume

In [7]:
# because PDF files are binary encoded and not plain text, we need to import a PDF Parser library, like pdfplumber
import pdfplumber

file_path2 = input("Please paste the fullpath to your .pdf resume file (eg: '/Users/your_name/Desktop/your_resume.pdf'): ").strip()

resume_string2 = " "
with pdfplumber.open(file_path2) as pdf:
    for page in pdf.pages:
        resume_string2 += page.extract_text() +"/n"

print(resume_string2[:500])

Please paste the fullpath to your .pdf resume file (eg: '/Users/your_name/Desktop/your_resume.pdf'):  /Users/nicholasjoseph/Desktop/ResumesEtc/nj_master_resume.pdf


Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

 Nicholas Joseph
Sales Engineer | Servant Leader | Customer Enthusiast
Email: nickpjoseph210@gmail.com | Phone: (210) 771-9853 | LinkedIn | GitHub
Career Summary
DATA ENGINEERING: 4 years experience utilizing proficiencies in Python, SQL, and data transformation
tools to engineer and implement data pipelines for data analysis and modeling.
MANAGEMENT: 7 yrs of management experience prioritizing and managing diverse teams of multiple
disciplines and specialties in both Agile and Waterfall environ


***
#### Step 2: Input the Desired Job Description
***

In [ ]:
job_desc_string = input("Please paste selected job descripton here: ")

***
#### Step 3: Input the Company Name and "Slug" It
> ##### Strip whitespace, get rid of stuff like apostrophes, and lowercase everything
> ##### Turns "Bill and Ted's Excellent Adventure Company" into "billandtedsexcellentadventurecompany"
> ##### Again, so user can better track their resumes
***

In [ ]:
# strip whitespace 
company_raw = input("Please enter the company's name: ").strip()
# cut out anything that's not a letter or number from the company name and lowercase everything
company_slugged = re.sub(r"[^a-zA-Z0-9_-]", "", company_raw.lower())

***
#### Step 4: Template Some Prompt Engineering
> ##### Because ChatGPT can talk to an ATS better than we can
***

In [ ]:
prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts
    - Select only the core competencies most relevant to the job description

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)
    - Please frame the results to prove I will add considerable value to their team.

3. **Summary Selection**:
    - Please tailor the best best Summary format for the job description and Recruiter expectations.
    - Please word the Summary so that it is abundantly clear how my skills and experience will lead to quick success within the role.

4. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

5. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.
        d.) **Compare intent** of job description with selected Summary and provide ideas to better tailor the Summary.
        e.) **Score** predicted match between resume and job description.

6. **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

# assign it a variable for use later on in the code
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 5: Generate the Resume with <b><span style="color:darkorchid">gpt-4o-mini</span></b>:
> ##### Sets up the api client, desginates the model, outlines the roles, and balances response creativity with accuracy.
> ##### More detailed notes available in original <b><span style="color:firebrick">rez_optimizer_chatgpt</span></b> notebook
***

In [ ]:
# set up client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
     
    temperature = 0.7
)

# Get our response
response_string = response.choices[0].message.content

***
#### Step 6: Display the <span style="color:darkorchid">ChatGPT</span>-Optimized Resume
***

In [ ]:
response_list = response_string.split("## Additional Suggestions")

display(Markdown(response_list[0]))

***
#### Step 7: Save the Resume In PDF and Markdown Using <span style="color:deeppink">Weasyprint</span> and <b>uuid</b>
***

In [ ]:
# 8-character unique id from uuid
unique_id = str(uuid.uuid4())[:8]
# Save and send files to gpt_resumes directory
output_pdf_file = f"gpt_resumes/{company_slugged}{unique_id}_optimized.pdf"
output_markdown_file = f"gpt_resumes/{company_slugged}{unique_id}_optimized.md"

In [ ]:
# Convert / save response to HTML so the web can read it using Weasyprint
html_content = markdown(response_list[0])
HTML(string=html_content).write_pdf(output_pdf_file)

In [ ]:
# Write and save as Markdown for ease of editing
with open(output_markdown_file, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

***
#### Step 7: Display Suggestions for Improvement Here for Easy Reference While Editing
> ##### <span style="color:red">Recall</span>: we split the '<span style="font-family:menlo"><b>response_list</span></b>' variable into two parts, [0] - the resume, and [1] - suggestions for improvement
***

In [ ]:
display(Markdown(response_list[1]))